# House Price Prediction using Machine Learning

This notebook builds a machine learning workflow for predicting house prices using real estate listing data.

The project covers:
- optional data collection from web listings,
- data loading and cleaning,
- exploratory data analysis,
- feature engineering,
- Random Forest regression,
- hyperparameter tuning,
- model evaluation,
- feature importance,
- and a simple prediction interface.

The goal is not only to train a model, but to show a complete data science pipeline from raw data to an interpretable prediction system.

## 1. Optional Data Collection

The following cells show how the original dataset was collected from online real estate listings.  
For a normal run of the notebook, this section can be skipped if the CSV file already exists.

**Important note for presentation:** in the video walkthrough, focus on the machine learning pipeline rather than the scraping details.

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import csv
import time

In [ ]:
# Define headers for HTTP requests
HEADERS = {
    "User-Agent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36',
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br",
    "DNT": "1",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

# Base URL and maximum pages
BASE_URL = "https://www.zillow.com/ne"
MAX_PAGES = 40  # Number of pages to scrape

# Output CSV file
OUTPUT_FILE = "zillow_scraped_data.csv"

# Data storage
scraped_data = []

# Function to save data to CSV
def save_to_csv(data, output_file):
    if not data:
        print("No data to save.")
        return

    keys = data[0].keys()
    with open(output_file, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=keys)
        writer.writeheader()
        writer.writerows(data)
    print(f"Data saved successfully to {output_file}")

# Start scraping
page = 1
while page <= MAX_PAGES:
    url = BASE_URL if page == 1 else f"{BASE_URL}/{page}_p"
    print(f"Fetching page {page}: {url}")

    # Fetch page content
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        content = response.content
    except requests.RequestException as e:
        print(f"Failed to fetch data from {url}: {e}")
        break

    # Parse JSON from the page
    try:
        soup = BeautifulSoup(content, 'html.parser')
        script_content = soup.find('script', id='__NEXT_DATA__')
        if script_content:
            json_content = script_content.string
            data = json.loads(json_content)
        else:
            print("Required script tag not found.")
            break
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        break

    # Extract house details
    try:
        house_details = data['props']['pageProps']['searchPageState']['cat1']['searchResults']['listResults']
        if not house_details:
            print(f"No more results found on page {page}.")
            break

        # Extract features for each house
        for detail in house_details:
            try:
                home_info = detail.get('hdpData', {}).get('homeInfo', {})
                photo_urls = [photo.get('url', '') for photo in detail.get('carouselPhotos', [])]

                # Extract and append house details
                house_data = {
                    "HOUSE URL": detail.get('detailUrl', 'N/A'),
                    "PHOTO URLs": ','.join(photo_urls),
                    "PRICE": detail.get('price', 'N/A'),
                    "FULL ADDRESS": detail.get('address', 'N/A'),
                    "CITY": home_info.get('city', 'N/A'),
                    "STATE": home_info.get('state', 'N/A'),
                    "ZIP CODE": home_info.get('zipcode', 'N/A'),
                    "NUMBER OF BEDROOMS": home_info.get('bedrooms', 'N/A'),
                    "NUMBER OF BATHROOMS": home_info.get('bathrooms', 'N/A'),
                    "HOUSE SIZE (sqft)": home_info.get('livingArea', 'N/A'),
                    "LOT SIZE": f"{home_info.get('lotAreaValue', 'N/A')} {home_info.get('lotAreaUnit', '')}",
                    "HOUSE TYPE": home_info.get('homeType', 'N/A').replace('_', ' ') if home_info.get('homeType') else 'N/A',
                    "HOME STATUS": home_info.get('homeStatus', 'N/A'),
                    "LATITUDE": home_info.get('latitude', 'N/A'),
                    "LONGITUDE": home_info.get('longitude', 'N/A'),
                    "STREET ADDRESS": home_info.get('streetAddress', 'N/A')
                }

                scraped_data.append(house_data)

            except Exception as e:
                print(f"Error processing house detail: {e}")
    except KeyError as e:
        print(f"Key error while extracting data: {e}")
        break

    print(f"Page {page} scraped successfully.")
    page += 1

    # Delay between requests to avoid being blocked
    time.sleep(3)  # Reduced delay for faster scraping; adjust if needed

# Save data to CSV
save_to_csv(scraped_data, OUTPUT_FILE)

print(f"Scraping completed. {len(scraped_data)} records collected.")

In [ ]:
# Export the collected data to a CSV file.
# This cell is only needed if the scraping cell above was executed successfully.

output_file = 'real_estate_datal.csv'

if len(scraped_data) > 0:
    print(f"Saving data to {output_file}...")
    try:
        with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=scraped_data[0].keys())
            writer.writeheader()
            writer.writerows(scraped_data)
        print(f"Data saved to {output_file} successfully.")
    except Exception as e:
        print(f"Failed to save data to CSV: {e}")
else:
    print("No scraped data found. If you already have the CSV file, continue from the loading section.")

import os
print(f"Current working directory: {os.getcwd()}")

## 2. Import Libraries and Load the Dataset

This section loads the Python libraries used for analysis and reads the real estate dataset from a CSV file.

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

In [ ]:
from pathlib import Path

# The original notebook used 'real_estate_datal.csv'.
# A fallback name is included in case the file is later renamed more cleanly.
csv_candidates = [
    'real_estate_datal.csv',
    'real_estate_data.csv',
    'zillow_scraped_data.csv'
]

csv_path = None
for candidate in csv_candidates:
    if Path(candidate).exists():
        csv_path = candidate
        break

if csv_path is None:
    raise FileNotFoundError(
        "CSV file not found. Put 'real_estate_datal.csv' or 'real_estate_data.csv' "
        "in the same folder as this notebook."
    )

real_estate_data = pd.read_csv(csv_path)
df = pd.DataFrame(real_estate_data)

print(f"Loaded dataset from: {csv_path}")
print(f"Dataset shape: {df.shape}")
df.head()

## 3. Data Dictionary

The dataset contains listing-level information about houses, including price, location, property size, house type, and geographic coordinates.

Main columns include:
- **PRICE:** the listed house price.
- **CITY / STATE / ZIP CODE:** location information.
- **NUMBER OF BEDROOMS / NUMBER OF BATHROOMS:** property characteristics.
- **HOUSE SIZE:** living area of the house.
- **LOT SIZE:** land size of the property.
- **HOUSE TYPE:** property category.
- **LATITUDE / LONGITUDE:** geographic coordinates.
- **PHOTO URLs / HOUSE URL / STREET ADDRESS:** listing information that is useful for browsing, but not directly used for model training.

In [ ]:
df.info()

## 4. Data Cleaning

Before training the model, the dataset needs to be cleaned.  
The price column is converted from text such as `$225,000` into a numeric value.  
The lot size column is also converted into square feet so all properties use the same unit.

In [ ]:
def clean_price(price):
    """Convert price text such as '$225,000' or '$225K' into a numeric value."""
    if pd.isna(price):
        return np.nan

    price = str(price).replace('$', '').replace(',', '').replace('+', '').strip()

    if 'K' in price:
        price = price.replace('K', '')
        return float(price) * 1000

    return float(price)

df['PRICE'] = df['PRICE'].apply(clean_price)
df.rename(columns={'PRICE': 'PRICE$'}, inplace=True)

df[['PRICE$']].head()

In [ ]:
def convert_acres_to_sqft(value):
    """Convert lot size values from acres or sqft text into square feet."""
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value in ['N/A', 'N/A ', '']:
        return np.nan

    if 'acres' in value:
        acres = float(value.split()[0])
        return acres * 43560

    if 'sqft' in value:
        return float(value.replace(' sqft', '').replace(',', ''))

    try:
        return float(value)
    except ValueError:
        return np.nan

df['LOT SIZE'] = df['LOT SIZE'].apply(convert_acres_to_sqft)
df.rename(columns={'LOT SIZE': 'LOT SIZE (sqft)'}, inplace=True)

df[['LOT SIZE (sqft)']].head()

## 5. Initial Dataset Checks

This section checks descriptive statistics, unique values, duplicate rows, and missing values.  
These checks help identify data quality issues before modeling.

In [ ]:
df.describe(include='all').T

In [ ]:
summary_checks = pd.DataFrame({
    "Unique Values": df.nunique(),
    "Missing Values": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2)
})

print(f"Duplicated rows: {df.duplicated().sum()}")
summary_checks

## 6. Handling Missing Values and Removing Unused Columns

Rows without photo URLs are removed because listings without photos may be incomplete or less reliable.  
Missing numerical values are replaced with the median because the median is less sensitive to outliers than the mean.

Some columns are removed because they are identifiers or browsing-related fields, not useful numerical features for the model.

In [ ]:
# Drop listings without photos
df = df.dropna(subset=['PHOTO URLs'])

# Replace missing numerical values with medians
columns_to_replace = ['NUMBER OF BEDROOMS', 'NUMBER OF BATHROOMS', 'HOUSE SIZE (sqft)']

for column in columns_to_replace:
    median_value = df[column].median()
    df[column] = df[column].fillna(median_value)

print(f"Dataset shape after missing-value handling: {df.shape}")
df[columns_to_replace].isnull().sum()

In [ ]:
# Drop columns that are not used for training
columns_to_drop = ['HOUSE URL', 'PHOTO URLs', 'STREET ADDRESS', 'STATE', 'HOME STATUS']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

# Add an ID column for easier tracking
df.insert(0, 'ID', range(1, len(df) + 1))

df.head()

## 7. Exploratory Data Analysis

Exploratory data analysis helps understand the dataset before modeling.  
The following visualizations show the distribution of prices, numerical feature behavior, geographic patterns, and correlations between features.

In [ ]:
# Price distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['PRICE$'], kde=True, bins=30)
plt.title('Distribution of House Prices')
plt.xlabel('Price ($)')
plt.ylabel('Number of listings')
plt.ticklabel_format(style='plain', axis='x')
plt.show()

# Price boxplot
plt.figure(figsize=(10, 4))
sns.boxplot(x=df['PRICE$'])
plt.title('House Price Outliers')
plt.xlabel('Price ($)')
plt.ticklabel_format(style='plain', axis='x')
plt.show()

In [ ]:
# Numerical feature boxplots using a cleaner layout
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != 'ID']

for column in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[column])
    plt.title(f'Outlier Check: {column}')
    plt.xlabel(column)
    plt.ticklabel_format(style='plain', axis='x')
    plt.show()

In [ ]:
# Price outlier detection using IQR
Q1 = df['PRICE$'].quantile(0.25)
Q3 = df['PRICE$'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = df[(df['PRICE$'] < lower_bound) | (df['PRICE$'] > upper_bound)]

print(f"Lower bound: ${lower_bound:,.2f}")
print(f"Upper bound: ${upper_bound:,.2f}")
print(f"Number of price outliers: {len(outliers_iqr)}")

outliers_iqr.head()

In [ ]:
# Geographic distribution of listings
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    df['LONGITUDE'],
    df['LATITUDE'],
    c=df['PRICE$'],
    cmap='viridis',
    alpha=0.65,
    s=45
)

plt.colorbar(scatter, label='Price ($)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Geographic Distribution of Listings Colored by Price')
plt.grid(True, alpha=0.25)
plt.show()

In [ ]:
# Relationship between key numerical features and price
selected_pairplot_cols = [
    'PRICE$',
    'NUMBER OF BEDROOMS',
    'NUMBER OF BATHROOMS',
    'HOUSE SIZE (sqft)',
    'LOT SIZE (sqft)'
]

pairplot_df = df[selected_pairplot_cols].dropna()

# Use a sample if the dataset is large to keep the plot readable
if len(pairplot_df) > 500:
    pairplot_df = pairplot_df.sample(500, random_state=42)

sns.pairplot(pairplot_df, diag_kind='kde', corner=True)
plt.suptitle('Pairwise Relationships Between Price and Key Features', y=1.02)
plt.show()

In [ ]:
# Correlation heatmap
numeric_df = df.select_dtypes(include=['number'])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": 0.8}
)

plt.title('Correlation Heatmap of Numerical Features')
plt.show()

# 8. Machine Learning

The machine learning task is a **regression problem** because the target variable is a continuous numeric value: house price.

The model will use property features such as size, bathrooms, lot size, city, and ZIP code to predict the listing price.

In [ ]:
df.info()

## 8.1 Feature Preparation

The lot size column is ensured to be numeric.  
The house type column is one-hot encoded because it is categorical.  
The city column is label encoded to convert city names into numeric values that the model can process.

A future improvement would be to use one-hot encoding or target encoding for location features, because label encoding can sometimes create artificial ordering between city names.

In [ ]:
df['LOT SIZE (sqft)'] = df['LOT SIZE (sqft)'].replace(['N/A ', ''])
df['LOT SIZE (sqft)'] = pd.to_numeric(df['LOT SIZE (sqft)']).fillna(0)

# One-Hot Encoding for 'HOUSE TYPE'
from sklearn.preprocessing import OneHotEncoder

if 'HOUSE TYPE' in df.columns:
    df = pd.get_dummies(df, columns=['HOUSE TYPE'], drop_first=True, dtype='int')

from sklearn.preprocessing import LabelEncoder

le_city = LabelEncoder()
df['CITY'] = le_city.fit_transform(df['CITY'])

df.head()

## 8.2 Simple Outlier Filtering

Very small or very large property values can make model training less stable.  
The following filter keeps the analysis focused on common property ranges.

In [ ]:
df = df[
    (df['HOUSE SIZE (sqft)'] >= 1000) & (df['HOUSE SIZE (sqft)'] <= 8000) &
    (df['NUMBER OF BEDROOMS'] <= 6) &
    (df['NUMBER OF BATHROOMS'] <= 5)
]

print(f"Dataset shape after outlier filtering: {df.shape}")
df.describe()

## 8.3 Feature Selection

The first feature importance check uses a Random Forest model to estimate which features are most useful for predicting price.

In [ ]:
features_names = [
    'NUMBER OF BEDROOMS',
    'NUMBER OF BATHROOMS',
    'HOUSE SIZE (sqft)',
    'ZIP CODE',
    'LOT SIZE (sqft)',
    'CITY',
    'LATITUDE',
    'LONGITUDE',
    'HOUSE TYPE_LOT',
    'HOUSE TYPE_MANUFACTURED',
    'HOUSE TYPE_MULTI FAMILY',
    'HOUSE TYPE_SINGLE FAMILY',
    'HOUSE TYPE_TOWNHOUSE'
]

# Keep only columns that exist in the current dataframe
features_names = [col for col in features_names if col in df.columns]

features = df[features_names]
target = df['PRICE$']

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model_imp = RandomForestRegressor(random_state=42)
model_imp.fit(features, target)

importances = model_imp.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': features_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

display(feature_importance_df)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance_df.head(10),
    x='Importance',
    y='Feature'
)
plt.title('Top 10 Feature Importances for House Price Prediction')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

## 8.4 Random Forest Regression

Random Forest is an ensemble learning technique that builds multiple decision trees and combines their predictions.  
It is useful for real estate price prediction because house prices often depend on non-linear relationships between features such as size, location, and property type.

In [ ]:
feature_dec = [
    'NUMBER OF BATHROOMS',
    'HOUSE SIZE (sqft)',
    'LOT SIZE (sqft)',
    'CITY',
    'ZIP CODE'
]

X = df[feature_dec]
y = df['PRICE$']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 8.5 Untuned Random Forest Model

The untuned model is trained first as a starting point.  
Its performance is measured using MAE, MSE, RMSE, and R².

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

model_untuned = RandomForestRegressor(random_state=42)
model_untuned.fit(X_train, y_train)

train_score = model_untuned.score(X_train, y_train)
test_score = model_untuned.score(X_test, y_test)

print(f"Random Forest Regressor Train R² Score: {train_score:.2f}")
print(f"Random Forest Regressor Test R² Score: {test_score:.2f}")

y_pred = model_untuned.predict(X_test)

mae_untuned = mean_absolute_error(y_test, y_pred)
mse_untuned = mean_squared_error(y_test, y_pred)
rmse_untuned = np.sqrt(mse_untuned)
r2_untuned = r2_score(y_test, y_pred)

print(f"MAE: {mae_untuned:.2f}")
print(f"MSE: {mse_untuned:.2f}")
print(f"RMSE: {rmse_untuned:.2f}")
print(f"R²: {r2_untuned:.2f}")

### Evaluation Metrics

- **MAE** measures the average absolute prediction error. It is easy to understand because it uses the same unit as the target price.
- **MSE** squares the errors, so larger mistakes are penalized more heavily.
- **RMSE** is the square root of MSE. It is also in the same unit as price and is easier to interpret than MSE.
- **R²** shows how much of the variation in house prices is explained by the model. Higher values usually indicate better predictive performance.

## 8.6 Hyperparameter Tuning with GridSearchCV

GridSearchCV tries multiple combinations of Random Forest parameters and uses cross-validation to select the best-performing model configuration.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': ['sqrt', 'log2']
}

model_v2 = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=model_v2,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

model_tuned = grid_search.best_estimator_
print("Best parameters found:", grid_search.best_params_)

y_train_pred = model_tuned.predict(X_train)
y_test_pred = model_tuned.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

mae_tuned = mean_absolute_error(y_test, y_test_pred)
mse_tuned = mean_squared_error(y_test, y_test_pred)
rmse_tuned = np.sqrt(mse_tuned)

print(f"Tuned Random Forest Train R²: {train_r2:.2f}")
print(f"Tuned Random Forest Test R²: {test_r2:.2f}")
print(f"MAE: {mae_tuned:.2f}")
print(f"MSE: {mse_tuned:.2f}")
print(f"RMSE: {rmse_tuned:.2f}")

cv_scores = cross_val_score(model_tuned, X, y, cv=5, scoring='r2')
print(f"Cross-Validation R²: {cv_scores.mean():.2f} ± {cv_scores.std():.2f}")

## 8.7 Model Comparison

The following table and charts compare the untuned and tuned Random Forest models.  
Lower MAE/RMSE is better, while higher R² is better.

In [ ]:
model_results = pd.DataFrame({
    'Model': ['Untuned Random Forest', 'Tuned Random Forest'],
    'MAE': [mae_untuned, mae_tuned],
    'RMSE': [rmse_untuned, rmse_tuned],
    'R² Score': [r2_untuned, test_r2]
})

display(model_results)

# Plot comparison in a clean format
results_long = model_results.melt(
    id_vars='Model',
    value_vars=['MAE', 'RMSE', 'R² Score'],
    var_name='Metric',
    value_name='Value'
)

for metric in ['MAE', 'RMSE', 'R² Score']:
    plt.figure(figsize=(8, 5))
    data = results_long[results_long['Metric'] == metric]
    ax = sns.barplot(data=data, x='Model', y='Value')
    plt.title(f'Model Comparison: {metric}')
    plt.xlabel('')
    plt.ylabel(metric)
    plt.xticks(rotation=10)

    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', padding=3)

    plt.tight_layout()
    plt.show()

### Model Evaluation Notes

The model comparison shows whether tuning improved the model's prediction error and R² score.

If the training score is much higher than the testing score, this may indicate overfitting.  
That means the model learned the training data very well, but does not generalize equally well to unseen houses.

This is common in real estate prediction because house prices depend on complex factors such as neighborhood quality, market timing, and property condition, which may not be fully captured in the available dataset.

## 9. Price Prediction Function

The following function allows a user to enter house features and receive a predicted price.  
The direct `input()` call is kept optional so the notebook does not pause when running all cells.

In [ ]:
def predict_price_manual():
    print("Enter the following details to predict house price:")

    house_size = float(input("HOUSE SIZE (sqft): "))
    lot_size = float(input("LOT SIZE (sqft): "))
    number_of_bathrooms = float(input("NUMBER OF BATHROOMS: "))
    zip_code = input("ZIP CODE: ")
    city = input("CITY: ")

    input_data = pd.DataFrame({
        'HOUSE SIZE (sqft)': [house_size],
        'LOT SIZE (sqft)': [lot_size],
        'NUMBER OF BATHROOMS': [number_of_bathrooms],
        'ZIP CODE': [zip_code],
        'CITY': [city]
    })

    if city in le_city.classes_:
        city_encoded = le_city.transform([city])[0]
    else:
        raise ValueError(f"City '{city}' not recognized. Available cities: {list(le_city.classes_)}")

    input_data['CITY'] = city_encoded
    input_data = input_data[feature_dec]

    predicted_price = model_tuned.predict(input_data)

    print(input_data)
    print(f"The predicted price for the house is: ${predicted_price[0]:,.2f}")

# Uncomment the next line to test manual prediction:
# predict_price_manual()

### Example Inputs

These are sample values that can be used to test the prediction function or the Gradio interface.

| Price Example | House Size | Lot Size | Bathrooms | ZIP Code | City |
|---|---:|---:|---:|---|---|
| $225,000 | 1566 | 9178 | 1 | 68506 | Lincoln |
| $180,000 | 768 | 6834 | 1 | 68105 | Omaha |
| $265,000 | 1560 | 6875 | 2 | 68124 | Omaha |
| $320,000 | 1832 | 6534 | 3 | 68007 | Bennington |
| $269,000 | 1269 | 7920 | 2 | 68801 | Grand Island |

## 10. Saving the Model

The trained model and city label encoder are saved using `joblib` so they can be reused later without retraining.

In [ ]:
import joblib

joblib.dump(le_city, 'city_label_encoder.pkl')
joblib.dump(model_tuned, 'house_price_model_tuned.pkl')

print("Model and encoder saved successfully.")

## 11. Optional Gradio Interface

The Gradio interface provides a simple user-friendly demo for entering house features and receiving a predicted price.

Set `RUN_GRADIO = True` only when you want to launch the interface.

In [ ]:
# !pip install gradio

In [ ]:
RUN_GRADIO = False

if RUN_GRADIO:
    import gradio as gr

    def predict_price(house_size, lot_size, num_bathrooms, zip_code, city):
        input_data = pd.DataFrame([{
            'HOUSE SIZE (sqft)': house_size,
            'LOT SIZE (sqft)': lot_size,
            'NUMBER OF BATHROOMS': num_bathrooms,
            'ZIP CODE': zip_code,
            'CITY': city
        }])

        if city in le_city.classes_:
            city_encoded = le_city.transform([city])[0]
        else:
            raise ValueError(f"City '{city}' not recognized. Available cities: {list(le_city.classes_)}")

        input_data['CITY'] = city_encoded
        input_data = input_data[feature_dec]

        predicted_price = model_tuned.predict(input_data)
        return f"${predicted_price[0]:,.2f}"

    inputs = [
        gr.Number(label="House Size (sqft)", value=1566),
        gr.Number(label="Lot Size (sqft)", value=9178),
        gr.Number(label="Number of Bathrooms", value=1),
        gr.Textbox(label="Zip Code", value="68506"),
        gr.Dropdown(choices=list(le_city.classes_), label="City")
    ]

    outputs = gr.Textbox(label="Predicted Price")

    gr.Interface(
        fn=predict_price,
        inputs=inputs,
        outputs=outputs,
        title="House Price Predictor",
        description="Enter house features to estimate the listing price."
    ).launch()
else:
    print("Gradio interface skipped. Set RUN_GRADIO = True to launch it.")

# Conclusion

This project demonstrates a complete machine learning workflow for house price prediction.  
The notebook starts with real estate listing data, performs cleaning and preprocessing, explores the data visually, trains a Random Forest regression model, tunes its hyperparameters, evaluates performance, and prepares a simple prediction interface.

The strongest parts of this project are:
- working with real-world messy data,
- converting text-based features into usable numerical values,
- visualizing patterns and outliers,
- training and evaluating a regression model,
- comparing untuned and tuned performance,
- and preparing the model for simple user interaction.

Future improvements could include stronger location encoding, more property features, better outlier handling, log transformation of price, and additional models such as Gradient Boosting or XGBoost.